- concurrency: Dealing with many things at once
designing a program as independent tasks that can make progress in overlapping time. A concurrent program may run those tasks one-at-a-time , switching rapidly between them

--> one cook preparing a whole meal

- Parallelism: Doing many things as once
requires multiple CPU

--> three cook, cooking different dish simultaneously in real time

The GIL (Global Interpreter Lock)
What it is
The GIL is a mutex (a lock) used by CPython (the standard Python interpreter). It ensures that only one thread executes Python bytecode at any given moment, even on a multi-core machine.

Why it exists
CPython manages memory with reference counting. If multiple threads modified reference counts at the same time, counts could get corrupted, leading to leaked memory or premature deallocation. The GIL is the simplest way to make this safe. So:

The GIL makes CPython thread-safe at the cost of preventing true multi-core parallelism for pure-Python threads.

Its use (why you must understand it)
It dictates your concurrency strategy. Because of the GIL, CPU-bound threads do not run in parallel → you must use multiprocessing for heavy computation.
It does NOT block I/O. The GIL is released during I/O operations (network, disk, sleep), so threads remain great for I/O-bound work.
It explains "weird" slowdowns. Naively adding threads to a CPU-heavy task can make it slower due to lock contention.
Note: PEP 703 is making CPython optionally GIL-free (no-GIL / free-threaded builds), but for most current production code the GIL is still the default reality.



In [3]:
import time
import threading

def count_down(n):
    while n > 0:
        n -= 1

COUNT = 50_000_000

# --- Single-threaded ---
start = time.perf_counter()
count_down(COUNT)
print(f"Single thread : {time.perf_counter() - start:.2f}s")

# --- Two threads sharing the work ---
start = time.perf_counter()
t1 = threading.Thread(target=count_down, args=(COUNT // 2,))
t2 = threading.Thread(target=count_down, args=(COUNT // 2,))
t1.start(); t2.start()
t1.join();  t2.join()
print(f"Two threads   : {time.perf_counter() - start:.2f}s")


Single thread : 0.94s
Two threads   : 0.92s


In [4]:
# The GIL is released during time.sleep

import time
import threading

def do_wait(seconds, label):
    print(f"{label} start")
    time.sleep(seconds)            # GIL released here
    print(f"{label} done")

start = time.perf_counter()

# Sequential: 1 + 1 + 1 = 3 seconds
# do_wait(1, "A"); do_wait(1, "B"); do_wait(1, "C")

# Threaded: all run at once -> ~1 second total
threads = [threading.Thread(target=do_wait, args=(1, label))
           for label in ("A", "B", "C")]
for t in threads: t.start()
for t in threads: t.join()

print(f"Total: {time.perf_counter() - start:.2f}s")

A start
B start
C start
A doneC done

B done
Total: 1.01s


Takeaway: GIL + CPU-bound = no parallel speedup. GIL + I/O-bound = full concurrency speedup.